# QR Code Operations with Umi-OCR

This notebook demonstrates how to use Umi-OCR's QR code features:
- Reading/scanning QR codes and barcodes from images
- Generating QR code images from text

## Prerequisites

1. Umi-OCR must be running with HTTP service enabled
2. Python packages: requests, pillow

In [ ]:
# Install required packages
!pip install requests pillow

In [ ]:
import requests
import base64
import json
from pathlib import Path
from PIL import Image
import io

## Configuration

In [ ]:
# Umi-OCR configuration
UMI_OCR_HOST = "127.0.0.1"
UMI_OCR_PORT = 1224
BASE_URL = f"http://{UMI_OCR_HOST}:{UMI_OCR_PORT}"

## Helper Functions

In [ ]:
def image_to_base64(image_path):
    """Convert an image file to base64 string."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def base64_to_image(base64_string):
    """Convert base64 string to PIL Image."""
    image_data = base64.b64decode(base64_string)
    return Image.open(io.BytesIO(image_data))

def pil_image_to_base64(pil_image):
    """Convert a PIL Image to base64 string."""
    buffer = io.BytesIO()
    pil_image.save(buffer, format='PNG')
    return base64.b64encode(buffer.getvalue()).decode('utf-8')

## 1. Reading QR Codes and Barcodes

Umi-OCR supports reading various types of codes:
- QR Code
- Barcodes (EAN-13, EAN-8, UPC-A, UPC-E, Code-128, Code-39, etc.)
- Data Matrix
- PDF417
- And more...

In [ ]:
def read_qrcode(image_base64):
    """Read QR code or barcode from an image.
    
    Args:
        image_base64: Base64 encoded image string
        
    Returns:
        Dictionary containing scan results
    """
    url = f"{BASE_URL}/api/qrcode"
    payload = {"base64": image_base64}
    
    try:
        response = requests.post(url, json=payload)
        return response.json()
    except Exception as e:
        return {'code': -1, 'data': str(e)}

# Example: Read QR code from an image file
qr_image_path = "qrcode_sample.png"  # Replace with your QR code image

if Path(qr_image_path).exists():
    # Convert to base64 and read
    qr_image_b64 = image_to_base64(qr_image_path)
    result = read_qrcode(qr_image_b64)
    
    print("QR Code Reading Result:")
    print(json.dumps(result, indent=2, ensure_ascii=False))
    
    # Display decoded content
    if result.get('code') == 100:
        print("\nDecoded Content:")
        for item in result.get('data', []):
            print(f"Type: {item.get('type', 'Unknown')}")
            print(f"Text: {item.get('text', '')}")
            print("-" * 50)
    elif result.get('code') == 101:
        print("No QR code or barcode detected in the image.")
else:
    print(f"Image file '{qr_image_path}' not found.")
    print("Generate a QR code first using the cells below, then try reading it.")

## 2. Generating QR Codes

Create QR code images from text:

In [ ]:
def generate_qrcode(text, size=None):
    """Generate a QR code image from text.
    
    Args:
        text: Text to encode in QR code
        size: Optional size (width and height) in pixels
        
    Returns:
        Dictionary containing the base64 encoded QR code image
    """
    url = f"{BASE_URL}/api/qrcode/text"
    payload = {"text": text}
    
    if size:
        payload["size"] = size
    
    try:
        response = requests.post(url, json=payload)
        return response.json()
    except Exception as e:
        return {'code': -1, 'data': str(e)}

# Example: Generate a QR code
qr_text = "Hello from Umi-OCR! Visit: https://github.com/hiroi-sora/Umi-OCR"

print(f"Generating QR code for: {qr_text}")
result = generate_qrcode(qr_text, size=300)

if result.get('code') == 100:
    # Get base64 image
    qr_base64 = result.get('data', '')
    
    # Convert to PIL Image and display
    qr_image = base64_to_image(qr_base64)
    display(qr_image)
    
    # Optionally save to file
    output_path = "generated_qrcode.png"
    qr_image.save(output_path)
    print(f"\nQR code saved to: {output_path}")
else:
    print(f"Error generating QR code: {result.get('data', 'Unknown error')}")

## 3. Generate and Read Workflow

Let's create a complete workflow: generate a QR code and then read it back:

In [ ]:
# Step 1: Generate QR code
test_text = "Umi-OCR is awesome!"
print(f"Original text: {test_text}")

gen_result = generate_qrcode(test_text, size=200)

if gen_result.get('code') == 100:
    qr_base64 = gen_result.get('data')
    
    # Display generated QR code
    qr_img = base64_to_image(qr_base64)
    print("\nGenerated QR Code:")
    display(qr_img)
    
    # Step 2: Read the QR code back
    print("\nReading the generated QR code...")
    read_result = read_qrcode(qr_base64)
    
    if read_result.get('code') == 100:
        decoded_text = read_result.get('data', [])[0].get('text', '')
        print(f"Decoded text: {decoded_text}")
        
        # Verify
        if decoded_text == test_text:
            print("✓ Success! Original and decoded text match.")
        else:
            print("✗ Warning: Decoded text differs from original.")
    else:
        print(f"Failed to read QR code: {read_result.get('data')}")
else:
    print(f"Failed to generate QR code: {gen_result.get('data')}")

## 4. Batch QR Code Generation

Generate multiple QR codes at once:

In [ ]:
def generate_multiple_qrcodes(text_list, size=200):
    """Generate multiple QR codes from a list of texts."""
    results = []
    
    for i, text in enumerate(text_list):
        result = generate_qrcode(text, size)
        if result.get('code') == 100:
            results.append({
                'index': i,
                'text': text,
                'image_base64': result.get('data'),
                'success': True
            })
        else:
            results.append({
                'index': i,
                'text': text,
                'success': False,
                'error': result.get('data')
            })
    
    return results

# Example: Generate QR codes for multiple URLs
urls = [
    "https://github.com/hiroi-sora/Umi-OCR",
    "https://github.com/hiroi-sora/Umi-OCR/releases",
    "https://github.com/hiroi-sora/Umi-OCR/issues"
]

print("Generating multiple QR codes...")
qr_results = generate_multiple_qrcodes(urls, size=150)

# Display all generated QR codes
for result in qr_results:
    if result['success']:
        print(f"\nQR Code {result['index'] + 1}: {result['text']}")
        img = base64_to_image(result['image_base64'])
        display(img)
    else:
        print(f"\nFailed to generate QR code {result['index'] + 1}: {result['error']}")

## 5. Reading Multiple Codes from One Image

Umi-OCR can detect multiple QR codes or barcodes in a single image:

In [ ]:
# Create an image with multiple QR codes
from PIL import Image

# Generate several QR codes
qr_codes = generate_multiple_qrcodes(
    ["Code 1", "Code 2", "Code 3"],
    size=150
)

# Create a combined image
if all(qr['success'] for qr in qr_codes):
    # Convert to PIL images
    images = [base64_to_image(qr['image_base64']) for qr in qr_codes]
    
    # Combine horizontally
    total_width = sum(img.width for img in images)
    max_height = max(img.height for img in images)
    
    combined = Image.new('RGB', (total_width, max_height), 'white')
    x_offset = 0
    for img in images:
        combined.paste(img, (x_offset, 0))
        x_offset += img.width
    
    print("Combined image with multiple QR codes:")
    display(combined)
    
    # Read all codes from the combined image
    combined_b64 = pil_image_to_base64(combined)
    read_result = read_qrcode(combined_b64)
    
    print("\nReading all QR codes from the combined image:")
    if read_result.get('code') == 100:
        codes_data = read_result.get('data', [])
        print(f"Found {len(codes_data)} code(s):")
        for i, code in enumerate(codes_data):
            print(f"{i + 1}. {code.get('text', '')} (Type: {code.get('type', 'Unknown')})")
    else:
        print(f"Error: {read_result.get('data')}")
else:
    print("Failed to generate some QR codes.")

## 6. Practical Example: URL Shortener with QR Codes

Create a simple tool to generate QR codes for URLs:

In [ ]:
def create_url_qr_card(url, title="", size=300):
    """Create a QR code for a URL with a title."""
    # Generate QR code
    result = generate_qrcode(url, size)
    
    if result.get('code') != 100:
        print(f"Error generating QR code: {result.get('data')}")
        return None
    
    # Get QR code image
    qr_img = base64_to_image(result.get('data'))
    
    # Create a card with title and QR code
    from PIL import ImageDraw, ImageFont
    
    card_width = size + 40
    card_height = size + 80
    card = Image.new('RGB', (card_width, card_height), 'white')
    draw = ImageDraw.Draw(card)
    
    # Add title
    if title:
        draw.text((20, 10), title, fill='black')
    
    # Add QR code
    card.paste(qr_img, (20, 40))
    
    # Add URL at bottom
    url_text = url if len(url) <= 40 else url[:37] + "..."
    draw.text((20, card_height - 25), url_text, fill='gray')
    
    return card

# Example usage
url = "https://github.com/hiroi-sora/Umi-OCR"
title = "Umi-OCR GitHub Repository"

qr_card = create_url_qr_card(url, title, size=250)
if qr_card:
    display(qr_card)
    
    # Save the card
    output_file = "url_qr_card.png"
    qr_card.save(output_file)
    print(f"\nQR card saved to: {output_file}")

## Summary

This notebook demonstrated:
- Reading QR codes and barcodes from images
- Generating QR codes from text
- Complete generate-and-read workflow
- Batch QR code generation
- Reading multiple codes from one image
- Creating practical QR code cards

## Supported Code Types

Umi-OCR supports 19 types of codes:
- QR Code
- Data Matrix
- PDF417
- Aztec
- EAN-13, EAN-8
- UPC-A, UPC-E
- Code-128, Code-39, Code-93
- Codabar
- And more...

## References

- [QR Code API Documentation](../../docs/http/api_qrcode.md)
- [Umi-OCR GitHub](https://github.com/hiroi-sora/Umi-OCR)